In [1]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


2. Pull code from github

In [2]:
import os

# Clone your repo
!git clone https://github.com/asifaliansaree/Trustworthy-AI-for-Dermatology-Imaging.git

# Confirm files are there
repo = "Trustworthy-AI-for-Dermatology-Imaging"
print("\nFiles in ham10000/src:")
for f in sorted(os.listdir(f"{repo}/ham10000/src")):
    print(f"  {f}")

print("\nFiles in ham10000/configs:")
for f in sorted(os.listdir(f"{repo}/ham10000/configs")):
    print(f"  {f}")

Cloning into 'Trustworthy-AI-for-Dermatology-Imaging'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 181 (delta 5), reused 22 (delta 4), pack-reused 154 (from 1)
Receiving objects: 100% (181/181), 126.72 MiB | 41.80 MiB/s, done.
Resolving deltas: 100% (62/62), done.

Files in ham10000/src:
  evaluate.py
  evaluate_test.py
  evaluate_tta.py
  losses.py
  model.py
  train.py

Files in ham10000/configs:
  baseline.yaml
  dry_run.yaml
  efficientnet_b0.yaml
  efficientnet_b1.yaml
  metadata_fusion.yaml


3. Set-up all path

In [3]:
import os, shutil

# ── Source paths (your Kaggle dataset) ───────────────────────
KAGGLE_DATA = "/kaggle/input/datasets/asifaliansaree/ham10000-images/data"

IMG_PART_1    = f"{KAGGLE_DATA}/HAM10000_images_part_1"
IMG_PART_2    = f"{KAGGLE_DATA}/HAM10000_images_part_2"
SPLIT_CSV     = f"{KAGGLE_DATA}/HAM10000_split.csv"
CLASS_WEIGHTS = f"{KAGGLE_DATA}/class_weights.npy"
METADATA_CSV  = f"{KAGGLE_DATA}/HAM10000_metadata.csv"

# ── Working directory ─────────────────────────────────────────
WORK = "/kaggle/working/project/ham10000/data"
os.makedirs(WORK, exist_ok=True)

# ── Symlink images (saves disk space, avoids copying 1.8 GB) ──
img1_dst = f"{WORK}/HAM10000_images_part_1"
img2_dst = f"{WORK}/HAM10000_images_part_2"

if not os.path.exists(img1_dst):
    os.symlink(IMG_PART_1, img1_dst)
if not os.path.exists(img2_dst):
    os.symlink(IMG_PART_2, img2_dst)

# ── Copy small files ──────────────────────────────────────────
shutil.copy(SPLIT_CSV,     f"{WORK}/HAM10000_split.csv")
shutil.copy(CLASS_WEIGHTS, f"{WORK}/class_weights.npy")
shutil.copy(METADATA_CSV,  f"{WORK}/HAM10000_metadata.csv")

# ── Verify everything ─────────────────────────────────────────
checks = {
    "Images part 1": img1_dst,
    "Images part 2": img2_dst,
    "Split CSV":     f"{WORK}/HAM10000_split.csv",
    "Class weights": f"{WORK}/class_weights.npy",
    "Metadata CSV":  f"{WORK}/HAM10000_metadata.csv",
}

print("=== Path verification ===\n")
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    all_ok = all_ok and exists
    print(f"{'✓' if exists else '✗'} {name}: {path}")

print(f"\n{'All paths OK — ready to train' if all_ok else 'MISSING PATHS — fix before training'}")

=== Path verification ===

✓ Images part 1: /kaggle/working/project/ham10000/data/HAM10000_images_part_1
✓ Images part 2: /kaggle/working/project/ham10000/data/HAM10000_images_part_2
✓ Split CSV: /kaggle/working/project/ham10000/data/HAM10000_split.csv
✓ Class weights: /kaggle/working/project/ham10000/data/class_weights.npy
✓ Metadata CSV: /kaggle/working/project/ham10000/data/HAM10000_metadata.csv

All paths OK — ready to train


4. Write kaggle configration

In [4]:
import yaml, os

os.makedirs("/kaggle/working/project/ham10000/configs", exist_ok=True)
os.makedirs("/kaggle/working/project/ham10000/checkpoints/efficientnet_b0", exist_ok=True)
os.makedirs("/kaggle/working/project/experiments/efficientnet_b0", exist_ok=True)

kaggle_cfg = {
    "seed": 42,
    "data": {
        "data_dir":    "/kaggle/working/project/ham10000/data",
        "num_workers": 2,
        "img_size":    224,
        "augment":     True,
    },
    "model": {
        "architecture": "efficientnet_b0",
        "num_classes":   7,
        "metadata_dim":  0,
        "pretrained":    True,
        "dropout":       0.4,
        "freeze_epochs": 3,
    },
    "train": {
        "epochs":                   25,
        "batch_size":               32,
        "lr":                       5e-5,
        "weight_decay":             1e-4,
        "scheduler":                "cosine",
        "gradient_accumulation":    1,
        "max_grad_norm":            1.0,
        "use_ema":                  True,
        "ema_decay":                0.9999,
        "early_stopping_patience":  7,
    },
    "loss": {
        "name":             "label_smoothing",
        "label_smoothing":  0.1,
    },
    "output": {
        "checkpoint_dir": "/kaggle/working/project/ham10000/checkpoints/efficientnet_b0",
    },
    "logging": {
        "experiment_name": "efficientnet_b0",
        "save_dir":        "/kaggle/working/project/experiments/efficientnet_b0",
        "checkpoint_dir":  "/kaggle/working/project/ham10000/checkpoints/efficientnet_b0",
    },
}

cfg_path = "/kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0.yaml"
with open(cfg_path, "w") as f:
    yaml.dump(kaggle_cfg, f, default_flow_style=False)

print(f"Config saved to: {cfg_path}")
print("\nConfig contents:")
print(yaml.dump(kaggle_cfg, default_flow_style=False))

Config saved to: /kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0.yaml

Config contents:
data:
  augment: true
  data_dir: /kaggle/working/project/ham10000/data
  img_size: 224
  num_workers: 2
logging:
  checkpoint_dir: /kaggle/working/project/ham10000/checkpoints/efficientnet_b0
  experiment_name: efficientnet_b0
  save_dir: /kaggle/working/project/experiments/efficientnet_b0
loss:
  label_smoothing: 0.1
  name: label_smoothing
model:
  architecture: efficientnet_b0
  dropout: 0.4
  freeze_epochs: 3
  metadata_dim: 0
  num_classes: 7
  pretrained: true
output:
  checkpoint_dir: /kaggle/working/project/ham10000/checkpoints/efficientnet_b0
seed: 42
train:
  batch_size: 32
  early_stopping_patience: 7
  ema_decay: 0.9999
  epochs: 25
  gradient_accumulation: 1
  lr: 5.0e-05
  max_grad_norm: 1.0
  scheduler: cosine
  use_ema: true
  weight_decay: 0.0001



5. Set-up Python path and verify import

In [5]:
import sys

REPO = "/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging"
sys.path.insert(0, f"{REPO}/ham10000/src")
sys.path.insert(0, f"{REPO}/ham10000")

# Test all imports
try:
    from model    import build_model, DermaNet
    print("✓ model.py")
except Exception as e:
    print(f"✗ model.py: {e}")

try:
    from losses   import build_loss
    print("✓ losses.py")
except Exception as e:
    print(f"✗ losses.py: {e}")

try:
    from dataset  import HAM10000Dataset
    print("✓ dataset.py")
except Exception as e:
    print(f"✗ dataset.py: {e}")

try:
    from metadata_encoder import MetadataEncoder
    print("✓ metadata_encoder.py")
except Exception as e:
    print(f"✗ metadata_encoder.py: {e}")

try:
    import evaluate
    print("✓ evaluate.py")
except Exception as e:
    print(f"✗ evaluate.py: {e}")

print("\nAll imports done.")

✓ model.py
✓ losses.py
✓ dataset.py
✓ metadata_encoder.py
✓ evaluate.py

All imports done.


6. Quick dataset sanity-check

In [6]:
from dataset import HAM10000Dataset

for split in ["train", "val", "test"]:
    ds = HAM10000Dataset(
        data_dir="/kaggle/working/project/ham10000/data",
        split=split,
        augment=(split == "train")
    )
    img, label = ds[0]
    print(f"[{split}] {len(ds)} images | "
          f"img shape: {img.shape} | label: {label}")

[train] 8017 images | mode=train
[train] 8017 images | img shape: torch.Size([3, 224, 224]) | label: 4
[val] 1009 images | mode=val
[val] 1009 images | img shape: torch.Size([3, 224, 224]) | label: 4
[test] 989 images | mode=test
[test] 989 images | img shape: torch.Size([3, 224, 224]) | label: 4


In [13]:
import yaml, os

os.makedirs("/kaggle/working/project/ham10000/configs", exist_ok=True)
os.makedirs("/kaggle/working/project/ham10000/checkpoints/efficientnet_b0_v2", exist_ok=True)
os.makedirs("/kaggle/working/project/experiments/efficientnet_b0_v2", exist_ok=True)

cfg = {
    "seed": 42,
    "data": {
        "data_dir":    "/kaggle/working/project/ham10000/data",
        "num_workers": 2,
        "img_size":    224,
        "augment":     True,
    },
    "model": {
        "architecture":  "efficientnet_b0",
        "num_classes":   7,
        "metadata_dim":  0,
        "pretrained":    True,
        "dropout":       0.4,
        "freeze_epochs": 0,
    },
    "train": {
        "epochs":                  25,
        "batch_size":              32,
        "lr":                      0.0001,
        "weight_decay":            0.0001,
        "scheduler":               "cosine",
        "gradient_accumulation":   1,
        "max_grad_norm":           1.0,
        "use_ema":                 False,
        "early_stopping_patience": 25,
    },
    "loss": {
        "name":            "label_smoothing",
        "label_smoothing": 0.1,
    },
    "output": {
        "checkpoint_dir": "/kaggle/working/project/ham10000/checkpoints/efficientnet_b0_v2",
    },
    "logging": {
        "experiment_name": "efficientnet_b0_v2",
        "save_dir":        "/kaggle/working/project/experiments/efficientnet_b0_v2",
        "checkpoint_dir":  "/kaggle/working/project/ham10000/checkpoints/efficientnet_b0_v2",
    },
}

path = "/kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0_v2.yaml"
with open(path, "w") as f:
    yaml.dump(cfg, f)

# Verify it was saved
assert os.path.exists(path), "FAILED — file not saved"
print(f"✓ Config saved: {path}")
print(f"  lr={cfg['train']['lr']}")
print(f"  freeze_epochs={cfg['model']['freeze_epochs']}")
print(f"  use_ema={cfg['train']['use_ema']}")

✓ Config saved: /kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0_v2.yaml
  lr=0.0001
  freeze_epochs=0
  use_ema=False


In [14]:
!python {REPO}/ham10000/src/train.py \
    --config /kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0_v2.yaml


Device      : cuda
Experiment  : efficientnet_b0_v2
Architecture: efficientnet_b0
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:252: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 25 epochs — loss=label_smoothing, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):
Epoch   1/25 | train=1.3154 | val=1.1768 | bal_acc=0.4478 | lr=9.96e-05 | 128s
  → New best (0.4478), saved.
/kaggle/working/Trustworthy-AI-for-De

In [15]:
import yaml, os

os.makedirs("/kaggle/working/project/ham10000/checkpoints/efficientnet_b0_v3", exist_ok=True)
os.makedirs("/kaggle/working/project/experiments/efficientnet_b0_v3", exist_ok=True)

cfg = {
    "seed": 42,
    "data": {
        "data_dir":    "/kaggle/working/project/ham10000/data",
        "num_workers": 2,
        "img_size":    224,
        "augment":     True,
    },
    "model": {
        "architecture":  "efficientnet_b0",
        "num_classes":   7,
        "metadata_dim":  0,
        "pretrained":    True,
        "dropout":       0.3,       # reduced from 0.4
        "freeze_epochs": 0,
    },
    "train": {
        "epochs":                  30,
        "batch_size":              32,
        "lr":                      2e-4,     # doubled from 1e-4
        "weight_decay":            1e-4,
        "scheduler":               "cosine",
        "gradient_accumulation":   1,
        "max_grad_norm":           1.0,
        "use_ema":                 False,
        "early_stopping_patience": 30,
    },
    "loss": {
        "name":            "cross_entropy",  # no label smoothing
    },
    "output": {
        "checkpoint_dir": "/kaggle/working/project/ham10000/checkpoints/efficientnet_b0_v3",
    },
    "logging": {
        "experiment_name": "efficientnet_b0_v3",
        "save_dir":        "/kaggle/working/project/experiments/efficientnet_b0_v3",
        "checkpoint_dir":  "/kaggle/working/project/ham10000/checkpoints/efficientnet_b0_v3",
    },
}

path = "/kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0_v3.yaml"
with open(path, "w") as f:
    yaml.dump(cfg, f)

print(f"✓ Config saved: {path}")
print(f"  lr={cfg['train']['lr']}")
print(f"  loss={cfg['loss']['name']}")
print(f"  dropout={cfg['model']['dropout']}")
print(f"  epochs={cfg['train']['epochs']}")

✓ Config saved: /kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0_v3.yaml
  lr=0.0002
  loss=cross_entropy
  dropout=0.3
  epochs=30


In [16]:
# Replace the heavy augmentation with lighter version
train_transform_fix = '''
    if split == "train" and augment:
        return T.Compose([
            T.Resize((256, 256)),
            T.RandomCrop(224),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2,
                          saturation=0.2, hue=0.05),
            T.RandomRotation(degrees=90),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
'''

dataset_path = f"{REPO}/ham10000/dataset.py"
with open(dataset_path, "r") as f:
    content = f.read()

old_aug = """    if split == "train" and augment:
        return T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.7, 1.0),
                                ratio=(0.9, 1.1)),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.RandomRotation(degrees=90),
            T.RandomAffine(degrees=0, shear=10),
            T.RandomPerspective(distortion_scale=0.2, p=0.3),
            T.ColorJitter(brightness=0.3, contrast=0.3,
                          saturation=0.2, hue=0.05),
            T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            T.RandomGrayscale(p=0.05),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        ])"""

new_aug = """    if split == "train" and augment:
        return T.Compose([
            T.Resize((256, 256)),
            T.RandomCrop(224),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2,
                          saturation=0.2, hue=0.05),
            T.RandomRotation(degrees=90),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])"""

if old_aug in content:
    content = content.replace(old_aug, new_aug)
    with open(dataset_path, "w") as f:
        f.write(content)
    print("✓ Augmentation simplified")
else:
    print("Block not found — manually check dataset.py")

✓ Augmentation simplified


In [18]:
!python {REPO}/ham10000/src/train.py \
    --config /kaggle/working/project/ham10000/configs/kaggle_efficientnet_b0_v3.yaml


Device      : cuda
Experiment  : efficientnet_b0_v3
Architecture: efficientnet_b0
Metadata    : False
EMA         : False
Freeze ep   : 0
[train] 8017 images | mode=train
[val] 1009 images | mode=val
[test] 989 images | mode=test
/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:252: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

Training for 30 epochs — loss=cross_entropy, scheduler=cosine

/kaggle/working/Trustworthy-AI-for-Dermatology-Imaging/ham10000/src/train.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):
Epoch   1/30 | train=0.8135 | val=0.6464 | bal_acc=0.5121 | lr=1.99e-04 | 111s
  → New best (0.5121), saved.
/kaggle/working/Trustworthy-AI-for-Derm

In [2]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

OUT = "/kaggle/working/figures"
os.makedirs(OUT, exist_ok=True)

# ── Reconstruct all logs from known terminal output ───────────

logs = {}

# ResNet-18 baseline — from your Week 3 Day 1 terminal output
logs["ResNet-18 baseline"] = pd.DataFrame({
    "epoch":       list(range(1, 21)),
    "train_loss":  [1.2297,0.9080,0.7725,0.7301,0.6584,0.5955,0.5583,
                    0.5146,0.4565,0.4230,0.4007,0.3679,0.3119,0.2905,
                    0.2819,0.2478,0.2406,0.2236,0.2179,0.2238],
    "val_loss":    [0.9273,0.9599,0.8957,0.7727,0.7851,0.8813,0.7096,
                    0.7006,0.7027,0.6521,0.6466,0.6169,0.6048,0.6158,
                    0.6083,0.6050,0.6350,0.6180,0.5847,0.6452],
    "val_bal_acc": [0.6486,0.6882,0.6880,0.7617,0.7087,0.7156,0.7110,
                    0.7337,0.7460,0.7798,0.7561,0.7587,0.7848,0.7714,
                    0.7964,0.7758,0.7705,0.7882,0.7902,0.7843],
})

# EfficientNet-B0 v2 — from your Kaggle terminal output
logs["EfficientNet-B0 v2 (lr=1e-4)"] = pd.DataFrame({
    "epoch":       list(range(1, 26)),
    "train_loss":  [1.3154,1.0644,1.0106,0.9804,0.9540,0.9310,0.9087,
                    0.8945,0.8867,0.8743,0.8588,0.8555,0.8435,0.8255,
                    0.8208,0.8062,0.7973,0.7895,0.7922,0.7782,0.7814,
                    0.7769,0.7719,0.7714,0.7781],
    "val_loss":    [1.1768,1.1160,1.1330,1.0739,1.0576,1.0616,1.0299,
                    1.0722,1.0611,1.0443,1.0547,1.0377,1.0295,1.0343,
                    1.0310,1.0332,1.0395,1.0321,1.0353,1.0425,1.0457,
                    1.0411,1.0425,1.0311,1.0327],
    "val_bal_acc": [0.4478,0.5128,0.5357,0.5598,0.6140,0.6064,0.6352,
                    0.6202,0.6145,0.6336,0.6179,0.6370,0.6347,0.6308,
                    0.6287,0.6431,0.6336,0.6630,0.6536,0.6378,0.6324,
                    0.6423,0.6338,0.6359,0.6535],
})

# EfficientNet-B0 v3 — from your Kaggle terminal output
logs["EfficientNet-B0 v3 (lr=2e-4)"] = pd.DataFrame({
    "epoch":       list(range(1, 31)),
    "train_loss":  [0.8135,0.5623,0.4891,0.4437,0.4012,0.3720,0.3466,
                    0.3114,0.2804,0.2641,0.2497,0.2230,0.1973,0.1867,
                    0.1663,0.1572,0.1368,0.1144,0.1269,0.1076,0.0990,
                    0.0862,0.0807,0.0790,0.0746,0.0616,0.0695,0.0575,
                    0.0660,0.0571],
    "val_loss":    [0.6464,0.5986,0.5845,0.5784,0.5727,0.4968,0.5198,
                    0.5126,0.5180,0.5331,0.5602,0.6318,0.5547,0.5996,
                    0.6241,0.6514,0.6311,0.6244,0.6356,0.7199,0.7403,
                    0.7325,0.7293,0.7404,0.7777,0.7471,0.7570,0.7531,
                    0.7457,0.7466],
    "val_bal_acc": [0.5121,0.5815,0.5897,0.5843,0.6068,0.6357,0.6627,
                    0.6623,0.6617,0.6834,0.6758,0.6765,0.6937,0.6754,
                    0.7058,0.6991,0.6851,0.7165,0.6992,0.6823,0.7026,
                    0.6999,0.7050,0.6925,0.7149,0.7088,0.7024,0.7095,
                    0.7035,0.7068],
})

print("All logs reconstructed:")
for name, df in logs.items():
    best = df["val_bal_acc"].max()
    print(f"  {name}: {len(df)} epochs | best={best:.4f}")

COLORS = {
    "ResNet-18 baseline":            "#2a78d6",
    "EfficientNet-B0 v2 (lr=1e-4)":  "#1baf7a",
    "EfficientNet-B0 v3 (lr=2e-4)":  "#e34948",
}

# ── Figure 1: Val balanced accuracy comparison ────────────────
fig, ax = plt.subplots(figsize=(11, 5))
for name, df in logs.items():
    ax.plot(df["epoch"], df["val_bal_acc"],
            color=COLORS[name], linewidth=2, label=name)
    best_idx = df["val_bal_acc"].idxmax()
    best_ep  = int(df.loc[best_idx, "epoch"])
    best_acc = df.loc[best_idx, "val_bal_acc"]
    ax.scatter(best_ep, best_acc,
               color=COLORS[name], s=80, zorder=5)
    ax.annotate(f"best={best_acc:.4f}\n(ep{best_ep})",
                xy=(best_ep, best_acc),
                xytext=(best_ep + 0.5, best_acc + 0.012),
                fontsize=8, color=COLORS[name],
                arrowprops=dict(arrowstyle="->",
                                color=COLORS[name], lw=0.8))
ax.axhline(y=0.65, color="#888780", linestyle=":",
           linewidth=1, alpha=0.7, label="0.65 go/no-go threshold")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Val balanced accuracy", fontsize=12)
ax.set_title("Validation balanced accuracy — ResNet-18 vs EfficientNet-B0",
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(linewidth=0.4, alpha=0.5)
ax.set_ylim(0.10, 0.90)
fig.tight_layout()
p1 = f"{OUT}/model_comparison_bal_acc.png"
fig.savefig(p1, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved → {p1}")

# ── Figure 2: Loss curves comparison ──────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
for name, df in logs.items():
    ax.plot(df["epoch"], df["train_loss"],
            color=COLORS[name], linewidth=1.5,
            linestyle="--", alpha=0.6,
            label=f"{name} — train")
    ax.plot(df["epoch"], df["val_loss"],
            color=COLORS[name], linewidth=2,
            label=f"{name} — val")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Train vs validation loss — all experiments",
             fontsize=13)
ax.legend(fontsize=8, ncol=2)
ax.grid(linewidth=0.4, alpha=0.5)
ax.set_ylim(0, 1.35)
fig.tight_layout()
p2 = f"{OUT}/model_comparison_loss.png"
fig.savefig(p2, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved → {p2}")

# ── Figure 3: Individual subplots ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, df) in zip(axes, logs.items()):
    color = COLORS[name]
    ax2   = ax.twinx()
    ax.plot(df["epoch"], df["train_loss"],
            color="#e34948", linewidth=2,
            linestyle="--", label="train loss")
    ax.plot(df["epoch"], df["val_loss"],
            color="#EF9F27", linewidth=2,
            label="val loss")
    ax2.plot(df["epoch"], df["val_bal_acc"],
             color="#2a78d6", linewidth=2.5,
             label="val bal-acc")
    best_acc = df["val_bal_acc"].max()
    best_ep  = int(df.loc[df["val_bal_acc"].idxmax(), "epoch"])
    ax2.axvline(best_ep, color="#888780",
                linestyle=":", linewidth=1.2, alpha=0.7)
    ax2.annotate(f"best={best_acc:.4f}\nep{best_ep}",
                 xy=(best_ep, best_acc),
                 xytext=(best_ep + 0.5, best_acc - 0.06),
                 fontsize=7.5, color="#2a78d6")
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Loss", fontsize=9, color="#888780")
    ax2.set_ylabel("Bal acc", fontsize=9, color="#2a78d6")
    ax.set_title(f"{name}\nbest val bal-acc={best_acc:.4f}",
                 fontsize=9.5)
    ax.grid(linewidth=0.4, alpha=0.4)
    lines1, lbl1 = ax.get_legend_handles_labels()
    lines2, lbl2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, lbl1 + lbl2,
              fontsize=7.5, loc="upper right")
fig.suptitle("Training curves — all experiments comparison",
             fontsize=13, y=1.01)
fig.tight_layout()
p3 = f"{OUT}/all_models_training_curves.png"
fig.savefig(p3, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved → {p3}")

print(f"\nAll 3 figures saved to {OUT}")
print("Download from Kaggle Output tab on the right sidebar")

All logs reconstructed:
  ResNet-18 baseline: 20 epochs | best=0.7964
  EfficientNet-B0 v2 (lr=1e-4): 25 epochs | best=0.6630
  EfficientNet-B0 v3 (lr=2e-4): 30 epochs | best=0.7165
Saved → /kaggle/working/figures/model_comparison_bal_acc.png
Saved → /kaggle/working/figures/model_comparison_loss.png
Saved → /kaggle/working/figures/all_models_training_curves.png

All 3 figures saved to /kaggle/working/figures
Download from Kaggle Output tab on the right sidebar
